# Comparison of feast networks
- individual Sundays of Advent
- st. Mary feasts (Annunciatio, Purificatio, )

Building blocks

In [2]:
import pycantus
import pycantus.data as data
from pycantus.filtration import Filter
import copy
import utils
import importlib

In [3]:
corpus = data.load_dataset('cantuscorpus_v1.0')

Loading chants and sources...
Data loaded!


In [4]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 888010
Number of sources in CantusCorpus v1.0: 2278


In [5]:
# Clean the corpus
# Drop doxology
doxo_filter = pycantus.filtration.Filter('doxo_filter')
doxo_filter.add_value_exclude('cantus_id', '909000')
corpus.apply_filter(doxo_filter)

# Drop fragments => sources with less than 100 chants
corpus.drop_small_sources_data(min_chants=100)

In [6]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 855146
Number of sources in CantusCorpus v1.0: 510


### Advent

In [7]:
# Build corpora for Sundays, e.g. "Dom. 1 Adventus"
advent_sundays_corpora = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(corpus)
    sunday_filter = Filter(f'{feast}_filter')
    sunday_filter.add_value_include('feast', feast)
    sunday_corpus.apply_filter(sunday_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_corpora[feast] = sunday_corpus
    print(f'{feast}: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus: 6447 chants, 245 sources
Dom. 2 Adventus: 4544 chants, 240 sources
Dom. 3 Adventus: 4876 chants, 251 sources
Dom. 4 Adventus: 4689 chants, 251 sources


In [8]:
# Bulit copora for advent sundays Lauds only
advent_sundays_lauds_corpora = {}
lauds_filter = Filter(f'lauds_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(advent_sundays_corpora[feast])
    lauds_filter.add_value_include('office', 'L')
    sunday_corpus.apply_filter(lauds_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_lauds_corpora[feast] = sunday_corpus
    print(f'{feast} Lauds: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus Lauds: 833 chants, 102 sources
Dom. 2 Adventus Lauds: 648 chants, 97 sources
Dom. 3 Adventus Lauds: 689 chants, 106 sources
Dom. 4 Adventus Lauds: 727 chants, 112 sources


### Networks

In [9]:
import networkx as nx

In [10]:
advent_sundays_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_network.edgelist')

Dom. 1 Adventus: 245 nodes, 13735 edges
Dom. 2 Adventus: 240 nodes, 13306 edges
Dom. 3 Adventus: 251 nodes, 13927 edges
Dom. 4 Adventus: 251 nodes, 14066 edges


In [11]:
importlib.reload(utils)

for G in advent_sundays_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 245 (3)
       Edges | 13,735 (0)
      Degree | 112.12 (227)
  Components | 97.6% (5)
  Clustering | 0.9426
Modularity from Louvain | 0.4286 (6)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 240 (0)
       Edges | 13,306 (0)
      Degree | 110.88 (230)
  Components | 98.8% (2)
  Clustering | 0.9752
Modularity from Louvain | 0.4121 (4)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 251 (1)
       Edges | 13,927 (0)
      Degree | 110.97 (233)
  Components | 97.2% (4)
  Clustering | 0.9603
Modularity from Louvain | 0.4415 (7)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 251 (0)
       Edges | 14,066 (0)
      Degree | 112.08 (237)
  Components | 98.4% (2)
  Clustering | 0.9652
Modularity from Louvain | 0.4952 (4)

--------------------------------------


In [12]:
import importlib
import utils
importlib.reload(utils)

G1 = advent_sundays_nets["Dom. 1 Adventus"]
G2 = advent_sundays_nets["Dom. 2 Adventus"]

utils.compare_edge_overlap(G1, G2, threshold = 0.2)

{'network_1': 'Dom. 1 Adventus',
 'network_2': 'Dom. 2 Adventus',
 'threshold': 0.2,
 'edges_1': 10750,
 'edges_2': 10869,
 'shared_edges': 9953,
 'all_edges': 11666,
 'edge_overlap': 0.8531630378878793}

In [ ]:
import pandas as pd
import itertools

rows = []

for feast1, feast2 in itertools.combinations(advent_sundays_nets.keys(), 2):
    result = utils.compare_edge_overlap(
        advent_sundays_nets[feast1],
        advent_sundays_nets[feast2],
        threshold = 0.2)
    rows.append(result)

overlap_df = pd.DataFrame(rows)
overlap_df

,network_1,network_2,threshold,edges_1,edges_2,shared_edges,all_edges,edge_overlap
0,Dom. 1 Adventus,Dom. 2 Adventus,0.2,10750,10869,9953,11666,0.853163
1,Dom. 1 Adventus,Dom. 3 Adventus,0.2,10750,11306,9688,12368,0.783312
2,Dom. 1 Adventus,Dom. 4 Adventus,0.2,10750,10612,8065,13297,0.606528
3,Dom. 2 Adventus,Dom. 3 Adventus,0.2,10869,11306,10033,12142,0.826305
4,Dom. 2 Adventus,Dom. 4 Adventus,0.2,10869,10612,8439,13042,0.647063
5,Dom. 3 Adventus,Dom. 4 Adventus,0.2,11306,10612,8941,12977,0.688988
